# Superfermion — Interactive Demo Notebook

This notebook validates Superfermion interactively, covering every major feature:
- **Circuits**: Build, draw, compile, serialize
- **JAX Autograd**: Gradients, JIT, Hessians
- **QML**: VQE, QAOA, QSVM, QNG, Kernels
- **QDL**: QResNet, QAttention
- **QLLM**: QuantumGPT
- **QEC**: Surface Code
- **Cloud/Security**: IBM, AWS, Auth, Multi-tenancy
- **Chemistry**: H2 Hamiltonian, UCCSD

In [ ]:
import superfermion as sf
import jax
import jax.numpy as jnp
import optax
print(f'Superfermion {sf.__version__}')
print(f'JAX {jax.__version__} ({jax.devices()[0].device_kind})')

## 1. Circuit API

In [ ]:
# Bell State
c = sf.Circuit(2).h(0).cx(0, 1)
print(c.draw())
print(f'Qubits: {c.n_qubits}, Gates: {c.gate_count}, Depth: {c.depth}')

In [ ]:
# Run and measure
result = sf.run(c, shots=1000)
print(f'Measurement results: {result.counts}')

In [ ]:
# Compilation: SWAP -> 3 CNOT
c_swap = sf.Circuit(2).swap(0, 1)
compiled = sf.compile(c_swap)
print(f'SWAP ({c_swap.gate_count} gates) -> Compiled ({compiled.gate_count} CNOTs)')
print(compiled.draw())

In [ ]:
# Serialization round-trip
j = c.to_json()
c2 = sf.Circuit.from_json(j)
qasm = c.to_qasm3()
print('QASM3 Export:')
print(qasm)

## 2. JAX Autograd

In [ ]:
# Gradient computation
c = sf.Circuit(1).rx(sf.param('theta'), 0)
f = sf.qml.circuit_to_jax(c, backend='jax')

def loss(t): return jnp.abs(f(t)[1])**2

g = jax.grad(loss)(jnp.array(1.0))
print(f'Gradient at theta=1.0: {g:.6f}')
print(f'Analytical: {jnp.sin(1.0) * jnp.cos(1.0)/2:.6f}')

In [ ]:
# Hessian
from superfermion.nn.quantum_layer import QuantumLayer
c = sf.Circuit(1).rx(sf.param('a'), 0).ry(sf.param('b'), 0)
model = QuantumLayer(n_qubits=1, ansatz=c)
params = model.init(jax.random.PRNGKey(0))
def loss(p): return model.apply(p)[0]
h = jax.hessian(loss)(params)
print('Hessian (2x2):')
print(h['params']['weights']['params']['weights'])

## 3. VQE — Ground State Energy

In [ ]:
from superfermion.observables.core import Hamiltonian, PauliString
from superfermion.algorithms.vqe import VQE

H = Hamiltonian([
    PauliString('ZI', 0.3), PauliString('IZ', 0.3),
    PauliString('ZZ', -0.5), PauliString('XX', 0.2)
])
ansatz = sf.Circuit(2)
for i in range(2):
    ansatz.ry(sf.param(f'ry{i}'), i)
    ansatz.rz(sf.param(f'rz{i}'), i)
ansatz.cx(0, 1)
for i in range(2):
    ansatz.ry(sf.param(f'ry2_{i}'), i)

vqe = VQE(ansatz, H, optimizer=optax.adam(0.05))
result = vqe.minimize(iterations=50)
print(f'Ground state energy: {result.optimal_value:.6f}')
print(f'Converged in {len(result.history)} iterations')

## 4. Chemistry — H2 Molecule

In [ ]:
from superfermion.chemistry import get_molecular_hamiltonian, uccsd_ansatz

H2 = get_molecular_hamiltonian('H2')
print(f'H2 Hamiltonian terms: {len(H2.terms)}')
for t in H2.terms:
    print(f'  {t.pauli_str}: {t.coeff:.4f}')

ansatz = uccsd_ansatz(n_qubits=2, n_electrons=2)
vqe = VQE(ansatz, H2, optimizer=optax.adam(0.1))
result = vqe.minimize(iterations=30)
print(f'\nH2 ground state energy: {result.optimal_value:.4f} Ha')

## 5. QEC — Surface Code

In [ ]:
from superfermion.qec.codes.surface import SurfaceCode

sc3 = SurfaceCode(distance=3)
print(sc3)
c = sc3.build_syndrome_extraction()
print(f'Syndrome circuit: {c.n_qubits} qubits, {c.gate_count} gates')

## 6. Cloud & Security

In [ ]:
from superfermion.runtime.arbiter import ResourceArbiter
from superfermion.runtime.specs import list_devices, get_spec
from superfermion.serve.auth import VAULT, check_qubit_limit

print('Available devices:', list_devices())
print(f'IBM Eagle: {get_spec("ibm_eagle").n_qubits} qubits')

arb = ResourceArbiter()
print(f'Route 5q: {arb.route(n_qubits=5)}')
print(f'Route 50q: {arb.route(n_qubits=50)}')

print(f'\nAuth tiers: {list(set(v["tier"] for v in VAULT.values()))}')
check_qubit_limit(10, 'free')  # passes
print('Free tier qubit check (10q): PASS')

## 6.5. Noise Modeling & ZNE (Calibration-Driven)

Build a `NoiseModel` from device calibration data, then
use Zero-Noise Extrapolation (ZNE) driven by real gate fidelities.

In [ ]:
from superfermion.pulse.calibration import CalibrationSet
from superfermion.mitigation import zne_with_calibration, calibration_based_noise_model

# Create calibration and extract noise parameters
cals = CalibrationSet('ibm_brisbane', dt=0.222)
cals.add_default_single_qubit(0)
cals.add_default_single_qubit(1)
cals.add_default_two_qubit(0, 1)

params = cals.extract_noise_params()
print(f"1Q fidelity: {params['avg_1q_fidelity']:.6f}")
print(f"2Q fidelity: {params['avg_2q_fidelity']:.6f}")
print(f"Depolarizing 1Q: {params['depolarizing_1q']:.6f}")

# Build NoiseModel from calibration
nm = cals.to_noise_model()
print(f"NoiseModel: {nm}")

# Quick calibrated model from backend name
nm2 = calibration_based_noise_model('ibm_eagle')
print(f"IBM Eagle NoiseModel: {nm2}")

# Calibration-driven ZNE
c = sf.Circuit(1).h(0)
def obs(sv): return float(sv[0].real)
result = zne_with_calibration(c, obs, cals, scale_factors=[1, 2, 3])
print(f"ZNE value: {result['zne_value']:.6f}")
print(f"Raw values: {[round(v, 4) for v in result['raw_values']]}")


## 6.6. Cloud Job Scheduler

Priority-queue distributed execution with batch submission
and dependency chains.

In [ ]:
from superfermion.runtime.scheduler import CloudScheduler, JobPriority, SchedulingPolicy

# Create scheduler
scheduler = CloudScheduler(max_workers=4)
scheduler.register_backend('jax', provider='local')

# Submit jobs with priority
c = sf.Circuit(1).h(0)

jid1 = scheduler.submit(c, backend='jax', priority=JobPriority.HIGH)
jid2 = scheduler.submit(c, backend='jax', priority=JobPriority.NORMAL)

# Wait for results
r1 = scheduler.wait_for(jid1, timeout=30)
r2 = scheduler.wait_for(jid2, timeout=30)

print(f"Job {jid1[:8]}... completed: {r1.counts}")
print(f"Job {jid2[:8]}... completed: {r2.counts}")

# Metrics
m = scheduler.metrics()
print(f"Total: {m['total_jobs']}, Completed: {m['completed']}")
print(f"Avg wait: {m['avg_wait_ms']:.1f}ms")

# Cleanup
scheduler.stop()


## 7. QuantumGPT

In [ ]:
from superfermion.qllm.transformer import QuantumGPT

c = sf.Circuit(2).ry(sf.param('a'), 0).ry(sf.param('b'), 1).cx(0, 1)
model = QuantumGPT(vocab_size=32, dim=2, n_layers=2, n_heads=1, seq_len=8, q_circuit=c)
tokens = jnp.array([[1, 5, 3, 7, 2, 6, 4, 0]])
params = model.init(jax.random.PRNGKey(0), tokens)
logits = model.apply(params, tokens)
next_token = int(jnp.argmax(jnp.real(logits[0, -1])))
print(f'Input tokens: {tokens[0].tolist()}')
print(f'Predicted next token: {next_token}')
print(f'Logits shape: {logits.shape}')

## 8. Visualization

In [ ]:
from superfermion.viz.core import bloch_angles, state_bar_chart

# Bloch sphere for |+> state
sv = jnp.array([1/jnp.sqrt(2), 1/jnp.sqrt(2)], dtype=complex)
angles = bloch_angles(sv)
print(f'|+> Bloch angles: theta={angles["theta"]:.4f}, phi={angles["phi"]:.4f}')

# Bell state probability chart
from superfermion.backends.jax_sim import JAXBackend
sim = JAXBackend()
bell = sf.Circuit(2).h(0).cx(0, 1)
sv_bell = sim.simulate(bell, [])
print('\nBell State Probabilities:')
print(state_bar_chart(sv_bell))

In [ ]:
print('\n' + '='*50)
print('  SUPERFERMION NOTEBOOK VALIDATION COMPLETE')
print('='*50)